# Variables indexadas y sumatorios

In [ ]:
%pip install docplex cplex


## Un modelo con índices

Consideramos tres proveedores, identificados mediante $I=\{1,2,3\}$. La variable continua $x_i$ representa la cantidad que adquirimos al proveedor $i$. Sus costes por unidad y cantidades máximas son:

| Proveedor $i$ | $1$ | $2$ | $3$ |
|---|---|---|---|
| Coste $c_i$ | $4$ | $2$ | $3$ |
| Cantidad máxima $u_i$ | $2$ | $3$ | $4$ |

Debemos comprar al menos $d=8$ unidades en total. El modelo es:

$$
\begin{aligned}
\min\quad & \sum_{i\in I}c_i x_i\\
\text{s.a}\quad & \sum_{i\in I}x_i\geq d,\\
& 0\leq x_i\leq u_i, && \forall i\in I.
\end{aligned}
$$

In [1]:
from docplex.mp.model import Model

# 1. Definir los índices y los datos
I = [1, 2, 3]

c = {1: 4, 2: 2, 3: 3}
u = {1: 2, 2: 3, 3: 4}
d = 8

# 2. Crear el modelo
mdl = Model(name="modelo_indexado")

# 3. Declarar las variables
x = {i: mdl.continuous_var(lb=0, name=f"x_{i}") for i in I}

# 4. Escribir la función objetivo
mdl.minimize(mdl.sum(c[i] * x[i] for i in I))

# 5. Añadir las restricciones
mdl.add_constraint(mdl.sum(x[i] for i in I) >= d)

for i in I:
    mdl.add_constraint(x[i] <= u[i])

# 6. Resolver y mostrar la solución
sol = mdl.solve()
print(sol)


solution for: modelo_indexado
objective: 22
status: OPTIMAL_SOLUTION(2)
x_1=1.000
x_2=3.000
x_3=4.000



## Ejercicio

Implementa y resuelve el siguiente modelo para $I=\{1,2,3\}$, utilizando variables indexadas y sumatorios:

$$
\begin{aligned}
\max\quad & \sum_{i\in I}c_i x_i\\
\text{s.a}\quad & \sum_{i\in I}a_i x_i\leq 10,\\
& 0\leq x_i\leq u_i, && \forall i\in I.
\end{aligned}
$$

Los datos son:

| Índice $i$ | $1$ | $2$ | $3$ |
|---|---|---|---|
| $c_i$ | $5$ | $4$ | $3$ |
| $a_i$ | $2$ | $1$ | $1$ |
| $u_i$ | $3$ | $4$ | $2$ |

Asigna al modelo el nombre `nuevo_modelo`.

In [ ]:
# Resuelva aquí


## Variables con dos índices

Consideramos los conjuntos $I=\{1,2\}$ y $J=\{1,2,3\}$, y variables continuas $x_{ij}$. Los coeficientes de la función objetivo son:

$$
C=\begin{pmatrix}2&5&4\\4&1&3\end{pmatrix}.
$$

El primer índice indica la fila y el segundo la columna. Además, utilizamos los datos:

$$
b_1=6,\quad b_2=7,\qquad d_1=4,\quad d_2=3,\quad d_3=5.
$$

El modelo es:

$$
\begin{aligned}
\min\quad & \sum_{i\in I}\sum_{j\in J}c_{ij}x_{ij}\\
\text{s.a}\quad & \sum_{j\in J}x_{ij}\leq b_i, && \forall i\in I,\\
& \sum_{i\in I}x_{ij}=d_j, && \forall j\in J,\\
& x_{ij}\geq 0, && \forall i\in I,\ \forall j\in J.
\end{aligned}
$$

In [2]:
from docplex.mp.model import Model

# 1. Definir los índices y los datos
I = [1, 2]
J = [1, 2, 3]

c = {
    (1, 1): 2, (1, 2): 5, (1, 3): 4,
    (2, 1): 4, (2, 2): 1, (2, 3): 3
}

b = {1: 6, 2: 7}
d = {1: 4, 2: 3, 3: 5}

# 2. Crear el modelo
mdl = Model(name="dos_indices")

# 3. Declarar las variables
x = {
    (i, j): mdl.continuous_var(lb=0, name=f"x_{i}_{j}")
    for i in I for j in J
}

# 4. Escribir la función objetivo
mdl.minimize(
    mdl.sum(c[i, j] * x[i, j] for i in I for j in J)
)

# 5. Añadir las restricciones
for i in I:
    mdl.add_constraint(mdl.sum(x[i, j] for j in J) <= b[i])

for j in J:
    mdl.add_constraint(mdl.sum(x[i, j] for i in I) == d[j])

# 6. Resolver y mostrar la solución
sol = mdl.solve()
print(sol)


solution for: dos_indices
objective: 27
status: OPTIMAL_SOLUTION(2)
x_1_1=4.000
x_1_3=1.000
x_2_2=3.000
x_2_3=4.000



## Ejercicio

Implementa y resuelve el siguiente modelo para $I=\{1,2\}$ y $J=\{1,2\}$, utilizando variables con dos índices y sumatorios:

$$
\begin{aligned}
\max\quad & \sum_{i\in I}\sum_{j\in J}p_{ij}x_{ij}\\
\text{s.a}\quad & \sum_{j\in J}x_{ij}\leq b_i, && \forall i\in I,\\
& \sum_{i\in I}x_{ij}\leq u_j, && \forall j\in J,\\
& x_{ij}\geq 0, && \forall i\in I,\ \forall j\in J.
\end{aligned}
$$

Los datos son:

$$
P=\begin{pmatrix}5&2\\3&4\end{pmatrix},\qquad
b_1=3,\quad b_2=4,\qquad u_1=2,\quad u_2=6.
$$

Asigna al modelo el nombre `ejercicio_dos_indices`.

In [ ]:
# Resuelva aquí


## Uso de condicionales

Consideramos $I=\{1,2,3\}$ y variables continuas $x_{ij}$ definidas únicamente para $i\neq j$. Los coeficientes son:

$$
c_{12}=2,\quad c_{13}=5,\quad c_{21}=4,\quad
c_{23}=3,\quad c_{31}=1,\quad c_{32}=6.
$$

El modelo es:

$$
\begin{aligned}
\min\quad & \sum_{i\in I}\sum_{j\in I:\,j\neq i}c_{ij}x_{ij}\\
\text{s.a}\quad & \sum_{j\in I:\,j\neq i}x_{ij}=1, && \forall i\in I,\\
& \sum_{i\in I:\,i\neq j}x_{ij}=1, && \forall j\in I,\\
& x_{ij}+2x_{ji}\leq 1.5, && \forall i,j\in I:\ i<j,\\
& x_{ij}\geq 0, && \forall i,j\in I:\ i\neq j.
\end{aligned}
$$

In [3]:
from docplex.mp.model import Model

# 1. Definir los índices y los datos
I = [1, 2, 3]

c = {
    (1, 2): 2, (1, 3): 5,
    (2, 1): 4, (2, 3): 3,
    (3, 1): 1, (3, 2): 6
}

# 2. Crear el modelo
mdl = Model(name="indices_condicionados")

# 3. Declarar las variables
x = {
    (i, j): mdl.continuous_var(lb=0, name=f"x_{i}_{j}")
    for i in I for j in I if i != j
}

# 4. Escribir la función objetivo
mdl.minimize(
    mdl.sum(
        c[i, j] * x[i, j]
        for i in I for j in I if i != j
    )
)

# 5. Añadir las restricciones
for i in I:
    mdl.add_constraint(
        mdl.sum(x[i, j] for j in I if j != i) == 1
    )

for j in I:
    mdl.add_constraint(
        mdl.sum(x[i, j] for i in I if i != j) == 1
    )

for i in I:
    for j in I:
        if i < j:
            mdl.add_constraint(x[i, j] + 2 * x[j, i] <= 1.5)

# 6. Resolver y mostrar la solución
sol = mdl.solve()
print(sol)


solution for: indices_condicionados
objective: 10.5
status: OPTIMAL_SOLUTION(2)
x_1_2=0.500
x_1_3=0.500
x_2_1=0.500
x_2_3=0.500
x_3_1=0.500
x_3_2=0.500



## Ejercicio

Implementa y resuelve el siguiente modelo para $I=\{1,2,3\}$, creando únicamente las variables cuyos índices cumplen $i<j$:

$$
\begin{aligned}
\max\quad & \sum_{i\in I}\sum_{j\in I:\,i<j}c_{ij}x_{ij}\\
\text{s.a}\quad & \sum_{i\in I}\sum_{j\in I:\,i<j}x_{ij}\leq 2,\\
& 0\leq x_{ij}\leq 1, && \forall i,j\in I:\ i<j.
\end{aligned}
$$

Las variables son continuas y los datos son $c_{12}=3$, $c_{13}=5$ y $c_{23}=4$. Asigna al modelo el nombre `ejercicio_condiciones`.

In [ ]:
# Resuelva aquí
